<a href="https://colab.research.google.com/github/Bayzid03/Synthetic-Medical-Records-GANs/blob/main/SayMyName.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

For building synthetic medical records using GANs, we’ll break it down into the following steps:

**Data Preprocessing**

**GAN Architecture**

**Training Loop**

**Evaluating Model Performance**

**Generating Synthetic Medical Records**

In [3]:
import pandas as pd

df = pd.read_csv("/content/Follow-up_Records.csv")

print(df.head())

   patient_id  visit_date  age_years  weight_kg   bmi  systolic_bp_mmHg  \
0  P-2025-001  2024-02-15         52       83.7  28.3               138   
1  P-2025-001  2024-03-15         52       83.4  28.2               147   
2  P-2025-001  2024-04-15         52       83.1  28.1               140   
3  P-2025-001  2024-05-15         52       83.0  28.1               136   
4  P-2025-001  2024-06-15         52       82.6  27.9               133   

   diastolic_bp_mmHg  heart_rate_bpm  body_temp_C  fasting_glucose_mg_dL  ...  \
0                 86              80         36.8                    137  ...   
1                 89              80         37.0                    140  ...   
2                 84              76         36.8                    122  ...   
3                 88              77         36.8                    112  ...   
4                 88              78         36.8                    101  ...   

   diet_quality_score_0_100  sleep_hours  exercise_sessions_pe

In [4]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import numpy as np

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

encoder = OneHotEncoder(sparse_output=False)
cat_encoded = encoder.fit_transform(df[cat_cols])

scaler = MinMaxScaler(feature_range=(-1, 1))
num_scaled = scaler.fit_transform(df[num_cols])

# combine processed data
data_processed = np.hstack((num_scaled, cat_encoded))

In [5]:
import torch
import torch.nn as nn

data_dim = data_processed.shape[1]
rand_noise_input = 64

# generator
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(rand_noise_input, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, data_dim),
            nn.Tanh()  # output in range [-1, 1]
        )
    def forward(self, z):
        return self.model(z)

# discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(data_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()  # probability of real/fake
        )
    def forward(self, x):
        return self.model(x)

In [6]:
from torch.utils.data import DataLoader, TensorDataset

# convert data to PyTorch tensors
real_data = torch.tensor(data_processed, dtype=torch.float32)
dataset = TensorDataset(real_data)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# initialize models
generator = Generator()
discriminator = Discriminator()

# optimizers
lr = 0.0002
optim_G = torch.optim.Adam(generator.parameters(), lr=lr)
optim_D = torch.optim.Adam(discriminator.parameters(), lr=lr)

# loss
criterion = nn.BCELoss()

epochs = 2000
for epoch in range(epochs):
    for real_batch, in loader:
        batch_size = real_batch.size(0)

        # labels for real and fake data
        real_labels = torch.ones((batch_size, 1))
        fake_labels = torch.zeros((batch_size, 1))

        # train discriminator
        z = torch.randn(batch_size, rand_noise_input)
        fake_data = generator(z)

        real_loss = criterion(discriminator(real_batch), real_labels)
        fake_loss = criterion(discriminator(fake_data.detach()), fake_labels)
        d_loss = (real_loss + fake_loss) / 2

        optim_D.zero_grad()
        d_loss.backward()
        optim_D.step()

        # train generator
        z = torch.randn(batch_size, rand_noise_input)
        fake_data = generator(z)
        g_loss = criterion(discriminator(fake_data), real_labels)  # want fake to be real

        optim_G.zero_grad()
        g_loss.backward()
        optim_G.step()

    if epoch % 200 == 0:
        print(f"Epoch [{epoch}/{epochs}]  D_loss: {d_loss.item():.4f}  G_loss: {g_loss.item():.4f}")

Epoch [0/2000]  D_loss: 0.6928  G_loss: 0.7358
Epoch [200/2000]  D_loss: 0.1433  G_loss: 2.5621
Epoch [400/2000]  D_loss: 0.0914  G_loss: 1.9978
Epoch [600/2000]  D_loss: 0.0736  G_loss: 1.7219
Epoch [800/2000]  D_loss: 0.5643  G_loss: 2.6156
Epoch [1000/2000]  D_loss: 0.1019  G_loss: 2.4407
Epoch [1200/2000]  D_loss: 0.1690  G_loss: 1.6252
Epoch [1400/2000]  D_loss: 0.0961  G_loss: 3.3998
Epoch [1600/2000]  D_loss: 0.0372  G_loss: 2.7605
Epoch [1800/2000]  D_loss: 0.1013  G_loss: 4.2060


In [8]:
# generate new synthetic data
z = torch.randn(10, rand_noise_input)  # 10 synthetic samples
synthetic_data_scaled = generator(z).detach().numpy()

# inverse transform
num_synthetic = scaler.inverse_transform(synthetic_data_scaled[:, :len(num_cols)])
cat_synthetic = encoder.inverse_transform(synthetic_data_scaled[:, len(num_cols):])

# combine into dataframe
synthetic_df = pd.DataFrame(num_synthetic, columns=num_cols)
synthetic_df[cat_cols] = cat_synthetic

print(synthetic_df)

   age_years  weight_kg        bmi  systolic_bp_mmHg  diastolic_bp_mmHg  \
0  52.997372  81.169327  27.336000        123.033112          81.789337   
1  52.079067  81.899460  27.776012        130.083145          79.033615   
2  52.999622  80.996910  27.312878        121.201401          81.562225   
3  52.999760  80.986870  27.310841        121.430023          81.448051   
4  52.070763  81.506989  27.684713        125.698860          76.461624   
5  52.999985  80.892433  27.303532        120.848793          82.624794   
6  52.059769  81.486565  27.687145        127.503685          76.868637   
7  52.873775  81.269737  27.451981        124.082321          78.973541   
8  52.999958  80.890610  27.304047        120.668716          81.165695   
9  52.989216  81.144371  27.341507        121.549904          78.693886   

   heart_rate_bpm  body_temp_C  fasting_glucose_mg_dL  \
0       77.242279    37.254509             117.564888   
1       74.892647    36.826221              91.275276   
2  